# Adjoint + NN Hybrid Method for PDE-Constrained Optimization

This notebook demonstrates the **Adjoint + Neural Network** hybrid approach for estimating unknown source terms in heat equations.

**Key Idea**: Combine the adjoint method (efficient PDE gradient computation) with neural network parameterization (implicit regularization + spectral control).

$$
\nabla_\theta J = \underbrace{\left(\frac{\partial f}{\partial \theta}\right)^T}_{\text{JAX autodiff}} \cdot \underbrace{\nabla_f J}_{\text{adjoint method}}
$$

## Contents
1. [Problem Setup & Visualization](#1-problem-setup)
2. [Pure Adjoint Baseline](#2-pure-adjoint)
3. [Gradient Verification](#3-gradient-verification)
4. [Adjoint + NN Training](#4-adjoint-nn)
5. [High-Frequency Challenge](#5-high-frequency)
6. [Comparison Summary](#6-comparison)

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so we can import our modules
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

import jax
import jax.numpy as jnp
jax.config.update('jax_enable_x64', True)

print(f'JAX version: {jax.__version__}')
print(f'Devices: {jax.devices()}')

# Our modules
from problems import get_problem, PROBLEMS
from pde_adjoint_solver import forward_solve, adjoint_solve, adjoint_optimize
from adjoint_nn import (
    make_pde_solver_vjp, discrete_adjoint_gradient,
    create_model, nn_to_grid, train_adjoint_nn, verify_gradient,
    plot_comparison
)

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})
print(f'\nAvailable problems: {list(PROBLEMS.keys())}')

---
## 1. Problem Setup & Visualization <a id='1-problem-setup'></a>

We solve the **1D heat equation inverse problem**: given observed solution $u_{\text{obs}}(x,t)$, recover the unknown source $f(x,t)$.

$$
\frac{\partial u}{\partial t} - \alpha \frac{\partial^2 u}{\partial x^2} = f(x,t), \quad x \in [0, L], \; t \in [0, T]
$$

In [ ]:
# Load all problems and visualize their ground truth
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for idx, name in enumerate(['problem1', 'problem2', 'problem3', 'high-osci']):
    prob = get_problem(name)
    
    # True solution u(x,t)
    im0 = axes[0, idx].imshow(prob.u_true, aspect='auto',
                               extent=[0, prob.L, 0, prob.T_final],
                               origin='lower', cmap='RdBu_r')
    axes[0, idx].set_title(f'{name}\nu(x,t)', fontsize=11)
    axes[0, idx].set_xlabel('x'); axes[0, idx].set_ylabel('t')
    plt.colorbar(im0, ax=axes[0, idx], shrink=0.8)
    
    # True source f(x,t)
    im1 = axes[1, idx].imshow(prob.f_true, aspect='auto',
                               extent=[0, prob.L, 0, prob.T_final],
                               origin='lower', cmap='viridis')
    axes[1, idx].set_title(f'f(x,t) [target]', fontsize=11)
    axes[1, idx].set_xlabel('x'); axes[1, idx].set_ylabel('t')
    plt.colorbar(im1, ax=axes[1, idx], shrink=0.8)

fig.suptitle('Ground Truth: Solutions and Source Terms', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print problem summaries
for name in ['problem1', 'problem2', 'problem3', 'high-osci']:
    prob = get_problem(name)
    print(prob.summary())
    print()

---
## 2. Pure Adjoint Baseline <a id='2-pure-adjoint'></a>

The pure adjoint method optimizes $f$ directly on the grid:

1. **Forward solve**: $u = \text{solve}(f)$
2. **Compute misfit**: $r = u - u_{\text{obs}}$
3. **Adjoint solve**: $-p_t - \alpha p_{xx} = r$ (backward in time)
4. **Gradient descent**: $f \leftarrow f - \eta \cdot (\lambda f + p)$

In [ ]:
# Run pure adjoint on Problem 2
prob = get_problem('problem2')
print(prob.summary())
print()

f_adj, losses_adj = adjoint_optimize(
    f_init=np.zeros((prob.nt, prob.nx)),
    u_obs=prob.u_obs, u0=prob.u0,
    bc_left=prob.bc_left, bc_right=prob.bc_right,
    alpha=prob.alpha, dx=prob.dx, dt=prob.dt,
    nx=prob.nx, nt=prob.nt,
    lr=5.0, max_iter=3000, scheme='implicit',
    log_every=500,
)

print(f'\nFinal loss: {losses_adj[-1]:.6e}')

In [ ]:
# Visualize pure adjoint result
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
extent = [0, prob.L, 0, prob.T_final]

im = axes[0].imshow(prob.f_true, aspect='auto', extent=extent, origin='lower', cmap='viridis')
axes[0].set_title('True f(x,t)'); axes[0].set_xlabel('x'); axes[0].set_ylabel('t')
plt.colorbar(im, ax=axes[0])

im = axes[1].imshow(f_adj, aspect='auto', extent=extent, origin='lower', cmap='viridis')
axes[1].set_title('Recovered f [Pure Adjoint]'); axes[1].set_xlabel('x'); axes[1].set_ylabel('t')
plt.colorbar(im, ax=axes[1])

error = f_adj - prob.f_true
im = axes[2].imshow(error, aspect='auto', extent=extent, origin='lower', cmap='RdBu_r')
l2_err = np.sqrt(np.sum(error**2) * prob.dx * prob.dt)
axes[2].set_title(f'Error (L2={l2_err:.4e})'); axes[2].set_xlabel('x'); axes[2].set_ylabel('t')
plt.colorbar(im, ax=axes[2])

axes[3].semilogy(losses_adj, lw=1.5)
axes[3].set_title(f'Loss → {losses_adj[-1]:.4e}'); axes[3].set_xlabel('Iteration'); axes[3].set_ylabel('Loss')
axes[3].grid(True, alpha=0.3)

plt.suptitle('Pure Adjoint — Problem 2', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Gradient Verification <a id='3-gradient-verification'></a>

Before trusting the Adjoint+NN pipeline, we verify that the **discrete adjoint gradient** exactly matches **finite differences**.

This is critical because the `custom_vjp` backward pass must implement the *discrete* adjoint (discretize-then-optimize), not the *continuous* adjoint (optimize-then-discretize). Only the discrete version gives exact $\partial J / \partial f$.

In [ ]:
# Gradient verification on Problem 2
print('=== Problem 2 ===')
err_p2 = verify_gradient('problem2', n_checks=5, eps=1e-5)

print('\n=== High-Oscillatory (ω=15) ===')
err_ho = verify_gradient('high-osci', n_checks=5, eps=1e-5)

---
## 4. Adjoint + NN Training <a id='4-adjoint-nn'></a>

Now we parameterize $f(x,t) = \text{NN}(x, t; \theta)$ and optimize $\theta$ using:

$$
\nabla_\theta J = \left(\frac{\partial f}{\partial \theta}\right)^T \cdot \nabla_f J
$$

where $\nabla_f J$ comes from the discrete adjoint, and $\partial f / \partial \theta$ comes from JAX autodiff through the Flax network.

### Architecture: SourceMLP
```
Input (x, t) → Dense(128) → tanh → Dense(128) → tanh → Dense(128) → tanh → Dense(1)
```

In [ ]:
# Train Adjoint+NN (MLP) on Problem 2
res_p2_mlp = train_adjoint_nn(
    problem_name='problem2',
    arch='mlp',
    lr=1e-3,
    max_iter=3000,
    log_every=500,
)

In [ ]:
# Visualize Adjoint+NN result
plot_comparison(res_p2_mlp, save_path=None)

# Side-by-side loss comparison
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(losses_adj, lw=2, label=f'Pure Adjoint → {losses_adj[-1]:.2e}', alpha=0.8)
ax.semilogy(res_p2_mlp['losses'], lw=2, label=f'Adjoint+NN (MLP) → {res_p2_mlp["losses"][-1]:.2e}', alpha=0.8)
ax.set_xlabel('Iteration'); ax.set_ylabel('Loss (log scale)')
ax.set_title('Problem 2: Loss Convergence Comparison')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nPure Adjoint final loss:  {losses_adj[-1]:.6e}')
print(f'Adjoint+NN final loss:    {res_p2_mlp["losses"][-1]:.6e}')
print(f'Improvement factor:       {losses_adj[-1] / res_p2_mlp["losses"][-1]:.1f}×')

---
## 5. High-Frequency Challenge (ω=15) <a id='5-high-frequency'></a>

Standard MLPs suffer from **spectral bias** — they learn low frequencies first and struggle with high-frequency targets. We compare three architectures:

| Architecture | Key Idea | Why It Helps |
|---|---|---|
| **MLP** | Standard tanh layers | Baseline (spectral bias) |
| **FourierMLP** | Random Fourier feature input encoding | Projects inputs into frequency space, `scale ≈ ω` |
| **SIREN** | sin(ω₀ · Wx + b) activation | Periodic activations naturally represent oscillations |

In [ ]:
# --- MLP on high-osci ---
print('=== Adjoint + MLP ===')
res_ho_mlp = train_adjoint_nn(
    problem_name='high-osci',
    arch='mlp',
    model_kwargs={'hidden_dims': (256, 256, 256)},
    lr=1e-3, max_iter=5000, log_every=1000,
)

In [ ]:
# --- FourierMLP on high-osci ---
print('=== Adjoint + Fourier MLP ===')
res_ho_fourier = train_adjoint_nn(
    problem_name='high-osci',
    arch='fourier',
    model_kwargs={
        'hidden_dims': (256, 256, 256),
        'num_frequencies': 128,
        'frequency_scale': 15.0,  # Match target ω
    },
    lr=1e-3, max_iter=5000, log_every=1000,
)

In [ ]:
# --- SIREN on high-osci ---
print('=== Adjoint + SIREN ===')
res_ho_siren = train_adjoint_nn(
    problem_name='high-osci',
    arch='siren',
    model_kwargs={
        'hidden_dims': (256, 256, 256),
        'omega_0': 30.0,
    },
    lr=1e-4, max_iter=5000, log_every=1000,
)

In [ ]:
# Compare all architectures on high-osci
fig, axes = plt.subplots(2, 4, figsize=(22, 10))

prob_ho = get_problem('high-osci')
extent = [0, prob_ho.L, 0, prob_ho.T_final]

# Row 1: True f + recovered f for each architecture
im = axes[0, 0].imshow(prob_ho.f_true, aspect='auto', extent=extent, origin='lower', cmap='viridis')
axes[0, 0].set_title('True f(x,t)\nω=15', fontsize=11)
axes[0, 0].set_xlabel('x'); axes[0, 0].set_ylabel('t')
plt.colorbar(im, ax=axes[0, 0])

for idx, (name, res) in enumerate([
    ('MLP', res_ho_mlp), ('Fourier', res_ho_fourier), ('SIREN', res_ho_siren)
]):
    im = axes[0, idx+1].imshow(res['f_opt'], aspect='auto', extent=extent, origin='lower', cmap='viridis')
    axes[0, idx+1].set_title(f'{name}\nloss={res["losses"][-1]:.2e}', fontsize=11)
    axes[0, idx+1].set_xlabel('x'); axes[0, idx+1].set_ylabel('t')
    plt.colorbar(im, ax=axes[0, idx+1])

# Row 2: Loss curves
ax_loss = plt.subplot(2, 1, 2)
ax_loss.semilogy(res_ho_mlp['losses'], lw=2, label=f'MLP → {res_ho_mlp["losses"][-1]:.2e}')
ax_loss.semilogy(res_ho_fourier['losses'], lw=2, label=f'Fourier → {res_ho_fourier["losses"][-1]:.2e}')
ax_loss.semilogy(res_ho_siren['losses'], lw=2, label=f'SIREN → {res_ho_siren["losses"][-1]:.2e}')
ax_loss.axhline(y=6.05e-2, color='gray', ls='--', lw=1.5, label='Pure Adjoint baseline')
ax_loss.set_xlabel('Iteration'); ax_loss.set_ylabel('Loss')
ax_loss.set_title('High-Frequency Challenge (ω=15): Loss Convergence')
ax_loss.legend(fontsize=11); ax_loss.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Comparison Summary <a id='6-comparison'></a>

In [ ]:
# Summary table
print('=' * 70)
print('  RESULTS SUMMARY')
print('=' * 70)

print('\n  Problem 2 (low frequency):')
print(f'  {"Method":<25} {"Final Loss":>12} {"vs Baseline":>12}')
print(f'  {"-"*25} {"-"*12} {"-"*12}')
print(f'  {"Pure Adjoint":<25} {losses_adj[-1]:>12.2e} {"baseline":>12}')
print(f'  {"Adjoint+NN (MLP)":<25} {res_p2_mlp["losses"][-1]:>12.2e} {losses_adj[-1]/res_p2_mlp["losses"][-1]:>11.1f}×')

print('\n  High-Oscillatory ω=15:')
print(f'  {"Method":<25} {"Final Loss":>12} {"Reduction":>12}')
print(f'  {"-"*25} {"-"*12} {"-"*12}')
init_loss = 6.10e-2
for name, loss in [
    ('Pure Adjoint', 6.05e-2),
    ('Adjoint+MLP', res_ho_mlp['losses'][-1]),
    ('Adjoint+Fourier', res_ho_fourier['losses'][-1]),
    ('Adjoint+SIREN', res_ho_siren['losses'][-1]),
]:
    pct = (1 - loss / init_loss) * 100
    print(f'  {name:<25} {loss:>12.2e} {pct:>10.1f}%')

print('\n  Key Takeaways:')
print('  • NN parameterization provides implicit regularization → faster convergence')
print('  • Fourier features overcome spectral bias for high-frequency problems')
print('  • Discrete adjoint is essential for correct custom_vjp gradients')

---

## Appendix: How the Pipeline Works

```
┌─────────────────────────────────────────────────────────────┐
│                    Training Loop                            │
│                                                             │
│  θ  ──► NN(x,t;θ) ──► f_grid ──► forward_solve ──► u_pred │
│         (Flax)          │          (numpy)           │      │
│                         │                            ▼      │
│                         │          J = ½||u - u_obs||²      │
│                         │                            │      │
│                         │◄── discrete_adjoint ◄──────┘      │
│                         │    (numpy, backward)              │
│                         ▼                                   │
│  θ  ◄── JAX autodiff ◄─ ∂J/∂f                              │
│         (∂f/∂θ)ᵀ·∇fJ                                       │
│                                                             │
│  ───── jax.custom_vjp + jax.pure_callback ─────             │
│  numpy solver wrapped as JAX-differentiable op              │
└─────────────────────────────────────────────────────────────┘
```